# v8 GRPO Candidate Policy

This notebook downloads `candidates_v7.csv` and the v8 SFT artifact from Hugging Face with `HF_TOKEN`, then runs constrained GRPO-style policy improvement over legal candidate groups. It uploads artifacts to `devaanshpa/orbit-wars-agent/v7/grpo`.

In [ ]:
from getpass import getpass
import os

if not os.environ.get("HF_TOKEN"):
    token = getpass("HF_TOKEN: ").strip()
    if not token:
        raise RuntimeError("HF_TOKEN is required because this notebook uploads v8 GRPO artifacts")
    os.environ["HF_TOKEN"] = token

print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
from pathlib import Path
import os

repo_root = Path.cwd()
if not (repo_root / "notebooks" / "v8" / "train_grpo_policy.py").exists() and (repo_root.parent.parent / "notebooks" / "v8" / "train_grpo_policy.py").exists():
    repo_root = repo_root.parent.parent

data_path = os.environ.get("CANDIDATES_CSV", "").strip()
remote_data = os.environ.get("V8_GRPO_DATA_REMOTE_PATH", "").strip()
if data_path:
    print("Using explicit local CANDIDATES_CSV override:", data_path)
elif remote_data:
    print("GRPO will download candidate CSV from Hugging Face:", remote_data)
else:
    print("GRPO will download the newest data/*/candidates_v7.csv from Hugging Face.")

sft = os.environ.get("V8_SFT_ARTIFACT", "").strip()
remote_sft = os.environ.get("V8_SFT_REMOTE_PATH", "v7/sft/model_weights_v8_sft.json").strip()
if sft:
    print("Using explicit local V8_SFT_ARTIFACT override:", sft)
else:
    os.environ["V8_SFT_REMOTE_PATH"] = remote_sft
    print("GRPO will download SFT artifact from Hugging Face:", remote_sft)

print("repo_root:", repo_root)

In [ ]:
import importlib.util
import sys

missing = [pkg for pkg in ("torch", "huggingface_hub", "matplotlib") if importlib.util.find_spec(pkg) is None]
if missing:
    print("Missing packages:", missing)
    print("Install with:")
    print(f"{sys.executable} -m pip install torch huggingface-hub matplotlib")
    raise RuntimeError("Install missing packages, restart the kernel if needed, then rerun this notebook.")

import torch
print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())

In [ ]:
import os
import subprocess
import sys

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

# Defaults tuned for a 1000-game both-sides dataset on Kaggle 2*T4.
# Override any value by setting the matching environment variable before this cell.
grpo_epochs = os.environ.get("V8_GRPO_EPOCHS", "120")
grpo_batch_groups = os.environ.get("V8_GRPO_BATCH_GROUPS", "160")
grpo_samples = os.environ.get("V8_GRPO_SAMPLES", "10")
grpo_temperature = os.environ.get("V8_GRPO_TEMPERATURE", "0.90")
grpo_lr = os.environ.get("V8_GRPO_LR", "0.00025")
grpo_weight_decay = os.environ.get("V8_GRPO_WEIGHT_DECAY", "0.00018")
grpo_kl_weight = os.environ.get("V8_GRPO_KL_WEIGHT", "0.065")
grpo_entropy_weight = os.environ.get("V8_GRPO_ENTROPY_WEIGHT", "0.012")
grpo_supervised_anchor = os.environ.get("V8_GRPO_SUPERVISED_ANCHOR", "0.14")
grpo_patience = os.environ.get("V8_GRPO_PATIENCE", "24")

print("GRPO config:", {
    "epochs": grpo_epochs,
    "batch_groups": grpo_batch_groups,
    "samples_per_group": grpo_samples,
    "temperature": grpo_temperature,
    "lr": grpo_lr,
    "weight_decay": grpo_weight_decay,
    "kl_weight": grpo_kl_weight,
    "entropy_weight": grpo_entropy_weight,
    "supervised_anchor": grpo_supervised_anchor,
    "patience": grpo_patience,
})

cmd = [
    sys.executable,
    str(repo_root / "notebooks" / "v8" / "train_grpo_policy.py"),
    "--export-dir",
    str(repo_root / "notebooks" / "v8" / "exports" / "grpo"),
    "--epochs",
    grpo_epochs,
    "--batch-groups",
    grpo_batch_groups,
    "--samples-per-group",
    grpo_samples,
    "--temperature",
    grpo_temperature,
    "--lr",
    grpo_lr,
    "--weight-decay",
    grpo_weight_decay,
    "--kl-weight",
    grpo_kl_weight,
    "--entropy-weight",
    grpo_entropy_weight,
    "--supervised-anchor",
    grpo_supervised_anchor,
    "--patience",
    grpo_patience,
    "--upload",
]
if os.environ.get("CANDIDATES_CSV"):
    cmd.extend(["--csv", os.environ["CANDIDATES_CSV"]])
if os.environ.get("V8_GRPO_DATA_REMOTE_PATH"):
    cmd.extend(["--data-remote-path", os.environ["V8_GRPO_DATA_REMOTE_PATH"]])
if os.environ.get("V8_SFT_ARTIFACT"):
    cmd.extend(["--sft-artifact", os.environ["V8_SFT_ARTIFACT"]])
if os.environ.get("V8_SFT_REMOTE_PATH"):
    cmd.extend(["--sft-remote-path", os.environ["V8_SFT_REMOTE_PATH"]])

print("Running:", " ".join(cmd))
proc = subprocess.Popen(cmd, cwd=str(repo_root), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
return_code = proc.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, cmd)